In [ ]:
import nfl_data_py as nfl
import pandas as pd

# Pull seasonal stats 2018-2023
print("Pulling player stats...")
stats = nfl.import_seasonal_data(years=list(range(2018, 2024)))

# Quick look at what shown
print(f"Shape: {stats.shape}")
print(f"\nColumns:\n{stats.columns.tolist()}")
print(f"\nFirst look:")
stats.head()

In [ ]:
# How many players and seasons is shown?
print(f"Total rows: {stats.shape[0]}")
print(f"Total columns: {stats.shape[1]}")
print(f"\nSeasons available: {sorted(stats['season'].unique())}")
print(f"\nMissing values per column:")
print(stats.isnull().sum()[stats.isnull().sum() > 0])

In [ ]:
# Pull roster info to get player names and positions
print("Pulling roster data...")
rosters = nfl.import_seasonal_rosters(years=list(range(2018, 2024)))

# Quick look
print(f"Shape: {rosters.shape}")
print(f"\nColumns:\n{rosters.columns.tolist()}")
rosters.head()

In [ ]:
# Keep only the columns needed from rosters
roster_clean = rosters[['player_id', 'season', 'player_name', 'position', 
                          'age', 'years_exp', 'draft_number', 'entry_year',
                          'team', 'weight', 'height']].copy()

# Drop duplicates (some players appear multiple times per season)
roster_clean = roster_clean.drop_duplicates(subset=['player_id', 'season'])

# Merge stats + roster on player_id and season
df = pd.merge(stats, roster_clean, on=['player_id', 'season'], how='inner')

# Quick check
print(f"Merged dataset shape: {df.shape}")
print(f"\nPositions available: {sorted(df['position'].unique())}")
print(f"\nSample players:")
df[['player_name', 'position', 'season', 'team', 'age']].head(10)

In [ ]:
# Pull contract data from nfl_data_py
print("Pulling contract data...")
contracts = nfl.import_contracts()

# Quick look
print(f"Shape: {contracts.shape}")
print(f"\nColumns:\n{contracts.columns.tolist()}")
contracts.head()

In [ ]:
# Keep only the columns we need from contracts
contracts_clean = contracts[['player', 'position', 'team', 'year_signed', 
                               'years', 'value', 'apy', 'guaranteed', 
                               'draft_overall']].copy()

# Rename columns to match our merged df
contracts_clean = contracts_clean.rename(columns={
    'player': 'player_name',
    'apy': 'aav'  # renaming to aav since that's the industry standard term
})

# Quick check
print(f"Contract data shape: {contracts_clean.shape}")
print(f"\nPositions in contracts: {sorted(contracts_clean['position'].unique())}")
print(f"\nYear range: {contracts_clean['year_signed'].min()} - {contracts_clean['year_signed'].max()}")
print(f"\nSample contracts:")
contracts_clean.head(10)

### Merging contracts to player-seasons

Contracts are only valid for the seasons that fall within their signed date — a player's
contract signed in 2023 shouldn't be matched to their 2019 season stats. Filtering to
`year_signed <= season` before deduplicating avoids this lookahead bias, ensuring the
model only ever sees contract information that existed at the time of that season's
on-field performance.

In [ ]:
# Merge stats+roster with contracts, keeping only contracts
# signed on or before the season they're matched to
master_df = pd.merge(df, contracts_clean,
                     on=['player_name', 'position'],
                     how='inner')

# Only keep contracts where year_signed <= the season (avoids lookahead bias)
master_df = master_df[master_df['year_signed'] <= master_df['season']]

# Keep the most recent active contract per player per season
master_df = master_df.sort_values('year_signed', ascending=False)
master_df = master_df.drop_duplicates(subset=['player_name', 'season'], keep='first')

print(f"Master dataset shape: {master_df.shape}")
print(f"\nSeason range: {master_df['season'].min()} - {master_df['season'].max()}")

In [ ]:
# Confirm no player appears more than once per season after the merge
duplicates = master_df.groupby(['player_name', 'season']).size().reset_index(name='count')
print(f"Max times a player appears in one season: {duplicates['count'].max()}")
print(f"\nSample of final dataset:")
master_df[['player_name', 'position', 'season', 'team_x', 'age',
           'aav', 'guaranteed', 'fantasy_points']].head(10)

In [ ]:
# Save master dataset to our data folder
master_df.to_csv('../data/master_df.csv', index=False)

print(f"✅ Master dataset saved!")
print(f"\nFinal summary:")
print(f"Total player-season records: {master_df.shape[0]}")
print(f"Total columns: {master_df.shape[1]}")
print(f"Positions: {sorted(master_df['position'].unique())}")
print(f"Seasons: {master_df['season'].min()} - {master_df['season'].max()}")
print(f"Unique players: {master_df['player_name'].nunique()}")